In [30]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import json

X_TEST_PATH = Path("../../artifacts/data/X_test.pkl")
X_TRAIN_STATS_PATH = Path("../../artifacts/data/X_train_stats.csv")

X_TRAIN_MOID_PATH = Path("../../artifacts/data/X_train_no_moid.pkl")
Y_TRAIN_MOID = Path("../../artifacts/data/y_train_moid.pkl")
X_TRAIN_MOID_STATS_PATH = Path("../../artifacts/data/X_train_no_moid_stats.pkl")
X_LOGS_MOID = Path("./logs/inference_events_moid.jsonl")
TRANSFORMER_MOID = Path("../../artifacts/column_transformer_moid.pkl")

X_TRAIN_PATH = Path("../../artifacts/data/X_train.pkl")
Y_TRAIN_PATH = Path("../../artifacts/data/y_train.pkl")

X_LOGS_PHA_PATH = Path("./logs/inference_events_pha.jsonl")
TRANSFORMER_PHA_PATH = Path("../../artifacts/best_model_columntransformer.pkl")





X_test = joblib.load(X_TEST_PATH).reset_index(drop=True)

X_train_stats = pd.read_csv(X_TRAIN_STATS_PATH).set_index('Unnamed: 0', drop=True)
X_train_stats.index.name = "stats"

X_train  = joblib.load(X_TRAIN_PATH)
y_train = joblib.load(Y_TRAIN_PATH)
model_pha = joblib.load('../../artifacts/best_model.pkl')
columntransformer_pha = joblib.load(TRANSFORMER_PHA_PATH)

X_train_moid = joblib.load(X_TRAIN_MOID_PATH)
y_train_moid = joblib.load(Y_TRAIN_MOID)
X_train_moid_stats = joblib.load(X_TRAIN_MOID_STATS_PATH)

model_moid = joblib.load("../../artifacts/moid_best_model.pkl")
columntransformer_moid = joblib.load(TRANSFORMER_MOID)


def inference_loader(path, key='features'):
    data = []
    i = 0
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if key == 'features':
                loaded = json.loads(line)["features"]
                data.append(loaded) 
                if i > 15000:
                    break
            elif key == 'prediction':
                  data.append(json.loads(line)[key][0])
                  if i > 15000:
                    break
            i += 1
       
    return data


moid_cur = pd.DataFrame(inference_loader(X_LOGS_MOID))
moid_cur_target = pd.Series(inference_loader(X_LOGS_MOID, key='prediction'), name="moid_au")


pha_cur = pd.DataFrame(inference_loader(X_LOGS_PHA_PATH))
pha_cur_target = pd.Series(inference_loader(X_LOGS_PHA_PATH, "prediction"), name="pha")
pha_ref = X_train.copy()

pha_cur.dropna(inplace=True)
pha_cur.reset_index(drop=True,inplace=True)


def nan_indices(X_train):
    nan_ind = []
    for col in X_train.columns.tolist():
        if X_train[col].isna().sum() > 0:
            nan_ind.extend(X_train[X_train[col].isna()].index.tolist())
    return nan_ind

nan_ind = nan_indices(X_train)

pha_ref.drop(nan_ind, axis=0, inplace=True)
pha_ref.reset_index(drop=True,inplace=True)

y_train.drop(nan_ind, inplace=True)
y_train.reset_index(drop=True,inplace=True)





In [31]:
moid_target = "moid_au"
moid_ref = X_train_moid.copy()
moid_ref[moid_target] = y_train_moid

moid_cur[moid_target] = moid_cur_target


pha_target = "pha"
pha_ref[pha_target] = y_train
pha_cur[pha_target] = pha_cur_target

print(pha_cur)

            H  diameter_km size_category class_code  eccentricity  \
0      12.179       44.082         Small        APO         0.760   
1      32.380       19.311         Small        APO         0.681   
2      28.078        8.085         Small        APO         0.394   
3      24.214       36.042         Small        APO         0.738   
4      39.312       21.643         Small        APO         0.647   
...       ...          ...           ...        ...           ...   
14997  33.000       42.477         Small        APO         0.178   
14998  20.043       13.710         Small        APO         0.970   
14999  12.612       33.584         Small        APO         0.624   
15000  37.992        7.711         Small        APO         0.469   
15001  32.265       10.458         Small        APO         0.480   

       semimajor_axis_au  inclination_deg  perihelion_distance_au  \
0                 34.214           21.547                   1.195   
1                260.130         

In [32]:
numerical_columns_pha = pha_ref.select_dtypes(include='number').columns.tolist()
categorical_columns_pha = pha_ref.select_dtypes(include='str').columns.tolist()

numerical_columns_moid = moid_ref.select_dtypes(include="number").columns.tolist()
categorical_columns_moid = moid_ref.select_dtypes(include=['str', 'category']).columns.tolist()


In [43]:
77/222

0.34684684684684686

In [51]:
import zipfile
import io
from datetime import datetime, time


from evidently.legacy.pipeline.column_mapping import ColumnMapping
from evidently import  Dataset, DataDefinition, BinaryClassification
from evidently.presets import DataDriftPreset, ClassificationPreset
from evidently.legacy.metric_preset import ClassificationPreset, RegressionPreset, TargetDriftPreset, DataDriftPreset
from evidently.metrics import RecallByLabel
from evidently.legacy.report import Report





In [34]:

classifier = model_pha
classifier.fit(columntransformer_pha.transform(pha_ref[numerical_columns_pha + categorical_columns_pha]), y_train)

regressor_moid = model_moid
regressor_moid.fit(columntransformer_moid.transform(moid_ref[numerical_columns_moid + categorical_columns_moid]), moid_ref[moid_target])

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [35]:
pha_ref_pred = classifier.predict(columntransformer_pha.transform(pha_ref))
pha_cur_pred = classifier.predict(columntransformer_pha.transform(pha_cur))

moid_ref_pred = regressor_moid.predict(columntransformer_moid.transform(moid_ref[numerical_columns_moid + categorical_columns_moid]))
moid_cur_pred = regressor_moid.predict(columntransformer_moid.transform(moid_cur[numerical_columns_moid + categorical_columns_moid]))
#moid_curr_pred = regressor_moid.prediction(columntransformer_moid.transform(X_moid_current))

pha_ref['prediction'] = pha_ref_pred
pha_cur['prediction'] = pha_cur_pred

moid_ref['prediction'] = moid_ref_pred
moid_cur['prediction'] = moid_cur_pred



/home/merkis/macaw_ml/near_earth_asteroid_predictor/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [36]:
pha_cur

,H,diameter_km,size_category,class_code,eccentricity,semimajor_axis_au,inclination_deg,perihelion_distance_au,aphelion_distance_au,orbital_period_days,moid_au,mean_motion_deg_day,condition_code,data_arc,pha,prediction
0,12.179,44.082,Small,APO,0.760,34.214,21.547,1.195,3.948,1919625.579,0.429,3.283,1,41595.420,False,False
1,32.380,19.311,Small,APO,0.681,260.130,179.932,0.642,353.008,933865.651,0.201,3.590,1,6735.230,False,False
2,28.078,8.085,Small,APO,0.394,349.504,8.933,0.600,659.642,2210304.794,0.727,3.281,1,34933.717,False,False
3,24.214,36.042,Small,APO,0.738,301.010,78.454,0.523,713.575,1776951.050,0.663,1.037,8,18073.588,False,False
4,39.312,21.643,Small,APO,0.647,114.087,93.528,0.577,123.267,1283623.423,0.771,0.936,8,30108.631,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14997,33.000,42.477,Small,APO,0.178,181.286,164.821,0.834,394.732,324249.830,0.132,0.112,1,5290.942,False,False
14998,20.043,13.710,Small,APO,0.970,189.389,191.634,0.916,59.479,1526848.587,0.772,0.889,8,41478.830,False,False
14999,12.612,33.584,Small,APO,0.624,33.457,88.028,0.084,628.550,275134.029,0.504,2.020,1,9456.894,False,False
15000,37.992,7.711,Small,APO,0.469,321.396,97.920,0.089,677.918,132117.912,0.303,2.898,8,5162.608,False,False


In [37]:
pha_ref['pha'].dtype, pha_cur['pha'].dtype
pha_ref['prediction'].dtype, pha_cur['prediction'].dtype

(dtype('bool'), dtype('bool'))

In [38]:
pha_cur[pha_cur["pha"] == pha_cur["prediction"]].count() / len(pha_cur)

pha_cur[(pha_cur["pha"] == True) & (pha_cur["prediction"] == True)].count() / pha_cur[pha_cur["pha"] == True].count()

H                         0.742475
diameter_km               0.742475
size_category             0.742475
class_code                0.742475
eccentricity              0.742475
semimajor_axis_au         0.742475
inclination_deg           0.742475
perihelion_distance_au    0.742475
aphelion_distance_au      0.742475
orbital_period_days       0.742475
moid_au                   0.742475
mean_motion_deg_day       0.742475
condition_code            0.742475
data_arc                  0.742475
pha                       0.742475
prediction                0.742475
dtype: float64

In [39]:

data_definition_pha = DataDefinition(
    classification=[BinaryClassification(
        target="pha",
        prediction_labels="prediction"
    )]

)







In [69]:
column_mapping = ColumnMapping()

column_mapping.task = 'classification'
column_mapping.target = 'pha'
column_mapping.prediction = 'prediction'
# column_mapping.pos_label = True
# column_mapping.categorical_features = numerical_columns_pha
# column_mapping.numerical_features = categorical_columns_pha





In [73]:
pha_cur['prediction'].unique()

array([False,  True])

In [70]:
categorical_performance = Report(metrics=[ClassificationPreset()],)
categorical_performance.run(current_data=pha_cur, reference_data=pha_ref, column_mapping=column_mapping)


In [71]:
categorical_performance.show()

ValueError: ClassificationClassSeparationPlot can be calculated only on binary probabilistic predictions